In [12]:
from dotenv import load_dotenv
load_dotenv()
from IPython.display import Markdown, display

In [2]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [38]:
prompt_short = ChatPromptTemplate.from_template(
    "Explain {short} in short."
)
prompt_long = ChatPromptTemplate.from_template(
    "Explain {long} in detail."
)

In [87]:
llm = init_chat_model(
    model="mistral-medium-latest",
    model_provider="mistralai",
    max_tokens=800,
)

In [7]:
parser = StrOutputParser()

## Sequence Runnables

In [8]:
seq_runnables = prompt_short | llm | parser

In [36]:
display(Markdown(seq_runnables.invoke({"short": "Lord of the Mysteries"})))

*Lord of the Mysteries* (also known as *Library of Heaven's Path* in Chinese) is a popular Chinese web novel written by **Cuttlefish That Loves Diving** (爱潜水的乌贼). It’s a **fantasy, adventure, and mystery** story set in a world where **mysteries** (supernatural phenomena) shape reality.

### **Short Summary:**
The protagonist, **Klein Moretti**, is an ordinary man in a world where **mysteries** grant people superhuman abilities. After a near-death experience, he gains the ability to **observe and replicate mysteries**, becoming a **"Seeker"**—someone who explores and deciphers the hidden truths

## Parallel Runnables

In [11]:
from langchain_core.runnables import RunnableParallel

In [62]:
prompt_parallel_short = ChatPromptTemplate.from_template(
    "Name authors of {topic1}, {topic2}."
)
prompt_parallel_long = ChatPromptTemplate.from_template(
    "Explain {topic3} in detail."
)

In [70]:
parallel_runnable = RunnableParallel({
    "short": prompt_parallel_short | llm | parser,
    "long": prompt_parallel_long | llm | parser,
})

In [64]:
parallel_output = parallel_runnable.invoke({"topic1": "Jujutsu Kaisen",
                                            "topic2": "Lord of the mysteries",
                                            "topic3": "naruto"})

The problem with the above code is that input get selected based on the keys only.

Input goes in both the pipeline as a whole instead of seperate values.

In [65]:
display(Markdown(parallel_output["short"]))

- Jujutsu Kaisen – **Gege Akutami**
- Lord of the Mysteries – **Cuttlefish That Loves Diving** (in Chinese: 爱潜水的乌贼)

In [66]:
display(Markdown(parallel_output["long"]))

*Naruto* is one of the most iconic and influential anime and manga series of all time, created by **Masashi Kishimoto**. It follows the journey of **Naruto Uzumaki**, a young ninja with dreams of becoming the **Hokage** (the leader of his village) to earn the respect of others. The story is a blend of **action, adventure, drama, and deep philosophical themes**, exploring concepts like **perseverance, friendship, the cycle of hatred, and the power of bonds**.

---

## **📜 Overview of Naruto**
### **🎭 Main Story Arcs**
The *Naruto* series is divided into two major parts:
1. **Nar

## Runnable Lambda

In [27]:
from langchain_core.runnables import RunnableLambda

In [80]:
prompt = {
    "short": {"topic1": "Lord of the Mysteries",
               "topic2": "Reverend Insanity"},
    "long": {"topic3": "Jujutsu Kaisen"}
}

In [81]:
prompt_lambda_short = ChatPromptTemplate.from_template(
    "Explain {topic1} and {topic2} in one sentence."
)
prompt_lambda_long = ChatPromptTemplate.from_template(
    "Explain {topic3} in detail."
)

Problem is that it short is not a keyword.


Even if the keyword was short still it will give one input which is dictionary:

{"topic1": "LOTM",
               "topic2": "Reverend Insanity"}

In [82]:
parallel_and_lambda_runnable = RunnableParallel({
    "short": prompt_lambda_short | llm | parser,
    "long": prompt_lambda_long | llm | parser,
})
lambda_output=parallel_and_lambda_runnable.invoke(prompt)

KeyError: "Input to ChatPromptTemplate is missing variables {'topic1', 'topic2'}.  Expected: ['topic1', 'topic2'] Received: ['short', 'long']\nNote: if you intended {topic1} to be part of the string and not a variable, please escape it with double curly braces like: '{{topic1}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "

Now using RunnableLambda we will specify the inputs for each pipeline

In [83]:
parallel_and_lambda_runnable = RunnableParallel({
    "short": RunnableLambda(lambda x:x ["short"]) | prompt_lambda_short | llm | parser,
    "long": RunnableLambda(lambda x:x ["long"]) | prompt_lambda_long | llm | parser,
})
lambda_output=parallel_and_lambda_runnable.invoke(prompt)

In [84]:
display(Markdown(lambda_output["short"]))

*Lord of the Mysteries* is a mystery-solving, magic-wielding detective novel where a reincarnated man unravels supernatural secrets in a Victorian-inspired world, while *Reverend Insanity* is a dark, ruthless xianxia cultivation story where a scheming, immortal alchemist manipulates fate and power to defy the heavens.

In [85]:
display(Markdown(lambda_output["long"]))

*Jujutsu Kaisen* (呪術廻戦, *Jujutsu Kaisen*, lit. "Sorcery Fight") is a popular Japanese manga and anime series written and illustrated by **Gege Akutami**. It blends **dark fantasy, supernatural action, and horror** with deep character development and intense fight scenes. Below is a **detailed breakdown** of the series, covering its **world, plot, characters, themes, and impact**.

---

## **🌍 World & Setting**
### **1. Jujutsu Society (呪術社会)**
- The world of *Jujutsu Kaisen* is a hidden reality where **curses (呪い,

# Runnables Passthrough

In [86]:
from langchain_core.runnables import RunnablePassthrough

In [99]:
code_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a expert software engineer with 15+ years of experience who writes code."),
    ("human", "Write a code about {what}.")
])

explain_prompt = ChatPromptTemplate.from_template(
    "You are a helpful mentor in field of software enginneering with 20+ years of expierence. Explaint the code:\n{code}"
)

In [100]:
seq1 = code_prompt | llm | parser

seq2 = RunnableParallel({
    "code": RunnablePassthrough(),
    "explaination": explain_prompt | llm | parser,
})

runnable = seq1 | seq2

In [ ]:
result = runnable.invoke({"what": "simple loop using recursive"})

In [102]:
display(Markdown(result["code"]))
print("\n\n")
display(Markdown(result["explaination"]))

Here's a simple example of a loop implemented using recursion in Python. This example demonstrates a recursive function that prints numbers from 1 to n, similar to a `for` loop:

```python
def recursive_loop(current, end):
    """
    A recursive function that prints numbers from `current` to `end`.

    Args:
        current (int): The current number to print.
        end (int): The end number (inclusive).
    """
    if current > end:
        return  # Base case: stop recursion when current exceeds end

    print(current)  # Print the current number
    recursive_loop(current + 1, end)  # Recursive call with next number

# Example usage: Print numbers from 1 to 5
recursive_loop(1, 5)
```

### Explanation:
1. **Base Case**: The recursion stops when `current > end`. This prevents infinite recursion.
2. **Recursive Case**: The function calls itself with `current + 1`, effectively incrementing the counter like a loop.
3. **Output**: This will print:
   ```
   1
   2
   3
   4
   5
   ```

### Key Points:
- Recursion replaces the loop's iteration by calling the function repeatedly.
- Each recursive call must progress toward the base case (here, `current` increases toward `end`).
- Without a base case, the recursion would run indefinitely (like an infinite loop).

### Equivalent Iterative Loop:
For comparison, here's the same logic using a `for` loop:
```python
for i in range(1, 6):
    print(i)
```

### Another Example: Sum of Numbers
Here's a recursive function to sum numbers from 1 to n (like a loop that accumulates a total):
```python
def recursive_sum(n, current=1, total=0):
    if current > n:
        return total
    return recursive_sum(n, current + 1, total + current)

print(recursive_sum(5))  # Output: 15 (1+2+3+4+5)
```

### **Detailed Explanation of the Recursive Loop in Python**

As a mentor with 20+ years of experience, I'll break this down in a way that helps you **truly understand recursion**—not just memorize syntax.

---

## **1. The Given Code: `recursive_loop(current, end)`**
```python
def recursive_loop(current, end):
    if current > end:
        return  # Base case: stop recursion
    print(current)
    recursive_loop(current + 1, end)  # Recursive call
```

### **How It Works (Step-by-Step)**
Let’s trace `recursive_loop(1, 5)`:

| **Call** | **current** | **end** | **Action** | **Output** |
|----------|------------|---------|------------|------------|
| 1        | 1          | 5       | `1 <= 5` → print `1`, call `recursive_loop(2, 5)` | `1` |
| 2        | 2          | 5       | `2 <= 5` → print `2`, call `recursive_loop(3, 5)` | `2` |
| 3        | 3          | 5       | `3 <= 5` → print `3`, call `recursive_loop(4, 5)` | `3` |
| 4        | 4          | 5       | `4 <= 5` → print `4`, call `recursive_loop(5, 5)` | `4` |
| 5        | 5          | 5       | `5 <= 5` → print `5`, call `recursive_loop(6, 5)` | `5` |
| 6        | 6          | 5       | `6 > 5` → **Base case reached**, return (no more calls) | - |

**Final Output:**
```
1
2
3
4
5
```

### **Key Concepts in Recursion**
1. **Base Case (`if current > end`)**
   - **Purpose:** Stops the recursion to prevent infinite calls.
   - **Analogy:** Like a loop’s termination condition (`i < n` in `for i in range(n)`).
   - **What happens if missing?**
     - Infinite recursion → **Stack Overflow** (Python crashes with `RecursionError`).

2. **Recursive Case (`recursive_loop(current + 1, end)`)**
   - **Purpose:** Moves toward the base case by modifying arguments.
   - **Analogy:** Like `i += 1` in a `while` loop.
   - **Must progress toward base case!**
     - If `current` decreases (e.g., `current - 1`), it may never reach `end` → infinite recursion.

3. **Call Stack (Behind the Scenes)**
   - Each recursive call **adds a new frame** to the stack.
   - When the base case is hit, the stack **unwinds** (returns to previous calls).
   - **Visualization:**
     ```
     recursive_loop(1,5)
     → recursive_loop(2,5)
       → recursive_loop(3,5)
         → recursive_loop(4,5)
           → recursive_loop(5,5)
             → recursive_loop(6,5) → returns
           → returns
         → returns
       → returns
     → returns
     ```

---

## **2. Comparison with Iterative Loop**
### **Recursive Version**
```python
def recursive_loop(current, end):
    if current > end:
        return
    print(current)
    recursive_loop(current + 1, end)
```

### **Iterative Version (for loop)**
```python
for